In [24]:
import numpy as np
import pandas as pd
import pickle
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch_geometric.nn import RGCNConv
from sklearn.preprocessing import StandardScaler
from pyspark.sql import functions as Func
from pyspark.sql.window import Window


OUTPUT_PATH = "dbfs:/your/output/path"   # ← change to your path
T_IN        = 12
T_OUT       = 3
HIDDEN_DIM  = 64
BATCH_SIZE  = 32
EPOCHS      = 50
LR          = 1e-3
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
WINDOW      = 6    # rolling window size
 
FEATURE_COLS = [
    "cpu_utilization",
    "memory_utilization",
    "providerRPC_MCR",
    "consumerRPC_MCR",
    "HTTP_MCR",
    "consumerMQ_MCR",
]
TARGET_COLS  = ["cpu_utilization", "memory_utilization", "HTTP_MCR"]
N_TARGETS    = len(TARGET_COLS)
N_FEATURES   = len(FEATURE_COLS)

In [ ]:
# !pip install torch-sparse --no-build-isolation

### Data Loading


In [25]:
OUTPUT_PATH="data_clean"

res = spark.read.parquet(f"{OUTPUT_PATH}/MSResource_clean")
rtq = spark.read.parquet(f"{OUTPUT_PATH}/MSRTQps_clean")
cg  = spark.read.parquet(f"{OUTPUT_PATH}/MSCallGraph_clean")

print(f"  MSResource  : {res.count():,} rows")
print(f"  MSRTQps     : {rtq.count():,} rows")
print(f"  MSCallGraph : {cg.count():,} rows")

  MSResource  : 138,619,760 rows
  MSRTQps     : 936,532 rows
  MSCallGraph : 506,254,796 rows


### Buildiing Node Index

In [26]:
nodes_df = (
    cg.select(Func.col("DM").alias("msname"))
    .union(cg.select(Func.col("UM").alias("msname")))
    .filter(Func.col("msname") != "EXTERNAL")
    .distinct()
    .toPandas()
)
nodes_df = nodes_df.reset_index(drop=True)
nodes_df["node_id"] = nodes_df.index
node_map = dict(zip(nodes_df["msname"], nodes_df["node_id"]))
N_NODES  = len(node_map)
print(f"  Total nodes : {N_NODES:,}")

[Stage 37:=====================================================>(333 + 3) / 336]

  Total nodes : 16,437


### Timestamp Alignment

In [27]:
res_agg = (
    res.groupBy("msname", "t_idx")
    .agg(
        Func.mean("cpu_utilization").alias("cpu_utilization"),
        Func.mean("memory_utilization").alias("memory_utilization"),
    )
)
 
# ── Use rt_direction == "UM" to avoid double counting ─────────────────
cg_dedup = cg.filter(
    (Func.col("UM") != "EXTERNAL") &
    (Func.col("rt_direction") == "UM")      # ← key: UM side only
)
 
cg_agg = (
    cg_dedup.groupBy("UM", "DM", "rpctype", "t_idx")
    .agg(
        Func.mean("rt_abs").alias("avg_rt"),
        Func.count("traceid").alias("call_count"),
        Func.mean("call_depth").alias("avg_depth")
          if "call_depth" in cg.columns
          else Func.lit(1.0).alias("avg_depth"),
    )
)
 
t_res = set(res_agg.select("t_idx").distinct().toPandas()["t_idx"].tolist())
t_rtq = set(rtq.select("t_idx").distinct().toPandas()["t_idx"].tolist())
t_cg  = set(cg_agg.select("t_idx").distinct().toPandas()["t_idx"].tolist())
 
common_t = sorted(t_res & t_rtq & t_cg)
T_STEPS  = len(common_t)
t_map    = {t: i for i, t in enumerate(common_t)}
print(f"  Common timesteps : {T_STEPS:,}")
 
res_agg = res_agg.filter(Func.col("t_idx").isin(common_t))
rtq_f   = rtq.filter(Func.col("t_idx").isin(common_t))
cg_agg  = cg_agg.filter(Func.col("t_idx").isin(common_t))
 

  Common timesteps : 720


### Bulding MULTI-RELATIONAL EDGE INDEX

In [28]:
edges_pd = cg_agg.toPandas()
edges_pd["src"] = edges_pd["UM"].map(node_map)
edges_pd["dst"] = edges_pd["DM"].map(node_map)
edges_pd = edges_pd.dropna(subset=["src", "dst"])
edges_pd["src"] = edges_pd["src"].astype(int)
edges_pd["dst"] = edges_pd["dst"].astype(int)
 
# ── Map rpctype → integer relation ID ─────────────────────────────────
rpc_types    = sorted(edges_pd["rpctype"].unique().tolist())
type_map     = {t: i for i, t in enumerate(rpc_types)}
N_RELATIONS  = len(rpc_types)
print(f"  rpctype values  : {rpc_types}")
print(f"  N relations     : {N_RELATIONS}")
 
edges_pd["edge_type"] = edges_pd["rpctype"].map(type_map)
 
# ── Static multi-relational edges (aggregate over all timesteps) ───────
edges_static = (
    edges_pd.groupby(["src", "dst", "edge_type"])
    .agg(
        avg_rt     = ("avg_rt",     "mean"),
        call_count = ("call_count", "mean"),
        avg_depth  = ("avg_depth",  "mean"),
    )
    .reset_index()
)
 
# PyG format
edge_index = torch.tensor(
    [edges_static["src"].tolist(),
     edges_static["dst"].tolist()],
    dtype=torch.long
)
edge_type = torch.tensor(
    edges_static["edge_type"].tolist(),
    dtype=torch.long
)
 
# Edge features [E, 3]
edge_feat_np  = edges_static[["avg_rt","call_count","avg_depth"]].values.astype(np.float32)
edge_scaler   = StandardScaler()
edge_feat_np  = edge_scaler.fit_transform(edge_feat_np)
edge_attr     = torch.tensor(edge_feat_np, dtype=torch.float)
 
print(f"  Total edges     : {edge_index.shape[1]:,}")
print(f"  Edge attr shape : {edge_attr.shape}")

  rpctype values  : ['db', 'http', 'mc', 'mq', 'rpc', 'userDefined']
  N relations     : 6
  Total edges     : 43,451
  Edge attr shape : torch.Size([43451, 3])


In [29]:
# ── Isolated node check ────────────────────────────────────────────────
nodes_with_edges = set(edge_index[0].tolist()) | set(edge_index[1].tolist())
isolated         = N_NODES - len(nodes_with_edges)
print(f"  Isolated nodes  : {isolated:,}")
 
# Remove isolated nodes
active_nodes  = sorted(nodes_with_edges)
node_remap    = {old: new for new, old in enumerate(active_nodes)}
N_ACTIVE      = len(active_nodes)
 
src_new = [node_remap[s.item()] for s in edge_index[0]]
dst_new = [node_remap[d.item()] for d in edge_index[1]]
edge_index = torch.tensor([src_new, dst_new], dtype=torch.long)
print(f"  Active nodes    : {N_ACTIVE:,}")

  Isolated nodes  : 1,815
  Active nodes    : 14,622


### Building Node feature matrix

In [30]:
res_pd = res_agg.toPandas()
rtq_pd = rtq_f.select(
    "msname", "t_idx",
    "providerRPC_MCR", "consumerRPC_MCR",
    "HTTP_MCR", "consumerMQ_MCR"
).toPandas()
 
feat_pd = res_pd.merge(rtq_pd, on=["msname","t_idx"], how="outer").fillna(0)
feat_pd["node_id"] = feat_pd["msname"].map(node_map)
feat_pd            = feat_pd.dropna(subset=["node_id"])
feat_pd["node_id"] = feat_pd["node_id"].map(lambda x: node_remap.get(int(x)))
feat_pd            = feat_pd.dropna(subset=["node_id"])
feat_pd["node_id"] = feat_pd["node_id"].astype(int)
feat_pd["t_int"]   = feat_pd["t_idx"].map(t_map)
feat_pd            = feat_pd.dropna(subset=["t_int"])
feat_pd["t_int"]   = feat_pd["t_int"].astype(int)
 
X = np.zeros((T_STEPS, N_ACTIVE, N_FEATURES), dtype=np.float32)
for _, row in feat_pd.iterrows():
    t = int(row["t_int"])
    n = int(row["node_id"])
    X[t, n, :] = [row[c] for c in FEATURE_COLS]
 
print(f"  Feature matrix shape : {X.shape}  [T, N, F]")

  Feature matrix shape : (720, 14622, 6)  [T, N, F]


### Normalize node features

In [32]:
train_end   = int(T_STEPS * TRAIN_RATIO)
node_scaler = StandardScaler()
X_flat      = X.reshape(-1, N_FEATURES)
node_scaler.fit(X_flat[: train_end * N_ACTIVE])
X_scaled    = node_scaler.transform(X_flat).reshape(X.shape)
print(f"  Scaler fit on first {train_end} timesteps")

  Scaler fit on first 503 timesteps


### Add rolling features

In [33]:
roll_mean = np.zeros_like(X_scaled)
roll_std  = np.zeros_like(X_scaled)
for t in range(T_STEPS):
    s            = max(0, t - WINDOW + 1)
    roll_mean[t] = X_scaled[s:t+1].mean(axis=0)
    roll_std[t]  = X_scaled[s:t+1].std(axis=0) + 1e-8
 
X_final = np.concatenate([X_scaled, roll_mean, roll_std], axis=-1)
F_FINAL = X_final.shape[-1]
print(f"  Final feature shape  : {X_final.shape}  [T, N, F={F_FINAL}]")
 
# Save to disk for memory-mapped loading
np.save("/tmp/X_final.npy", X_final)

  Final feature shape  : (720, 14622, 18)  [T, N, F=18]


### Sliding Windo dataset

In [34]:
TARGET_IDX = [FEATURE_COLS.index(c) for c in TARGET_COLS]
 
 
class TemporalGraphDataset(Dataset):
    def __init__(self, X_path, T_in, T_out, target_idx):
        self.X          = np.load(X_path, mmap_mode="r")  # memory-mapped
        self.T_in       = T_in
        self.T_out      = T_out
        self.target_idx = target_idx
 
    def __len__(self):
        return len(self.X) - self.T_in - self.T_out + 1
 
    def __getitem__(self, idx):
        x = torch.tensor(
            self.X[idx : idx + self.T_in].copy(),
            dtype=torch.float
        )                                                         # [T_in, N, F]
        y = torch.tensor(
            self.X[idx + self.T_in :
                   idx + self.T_in + self.T_out, :, self.target_idx].copy(),
            dtype=torch.float
        )                                                         # [T_out, N, 3]
        return x, y
 
 
dataset  = TemporalGraphDataset("/tmp/X_final.npy", T_IN, T_OUT, TARGET_IDX)
n_total  = len(dataset)
n_train  = int(n_total * TRAIN_RATIO)
n_val    = int(n_total * VAL_RATIO)
n_test   = n_total - n_train - n_val
 
# Sequential split — no shuffle for temporal data
train_ds = torch.utils.data.Subset(dataset, range(0, n_train))
val_ds   = torch.utils.data.Subset(dataset, range(n_train, n_train + n_val))
test_ds  = torch.utils.data.Subset(dataset, range(n_train + n_val, n_total))
 
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)
 
print(f"  Total  : {n_total}  Train : {n_train}  Val : {n_val}  Test : {n_test}")

  Total  : 706  Train : 494  Val : 105  Test : 107


### Define TGCN Model

In [35]:
print(f"MSResource  steps : {len(t_res):,}")
print(f"MSRTQps     steps : {len(t_rtq):,}")
print(f"MSCallGraph steps : {len(t_cg):,}")
print(f"Common      steps : {T_STEPS:,}")
print(f"Lost steps        : {max(len(t_res),len(t_rtq),len(t_cg)) - T_STEPS:,}")

MSResource  steps : 1,440
MSRTQps     steps : 721
MSCallGraph steps : 1,441
Common      steps : 720
Lost steps        : 721
